# Applicazione di ELIta al corpus r/Italia — keyword *notizie*

Questo notebook applica il lessico ELIta (originale e versioni ricalcolate) ai commenti raccolti da r/Italia con keyword **notizie**.

## Import e configurazione

In [1]:
import pandas as pd
import numpy as np
import emoji
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import silhouette_score
from pathlib import Path

CORPUS_CSV   = Path('corpus_Italia_notizie.csv')
TOKENS_CSV   = Path('tokens_Italia_notizie.csv')
ELITA_CSV    = Path('../Fase1/ELIta_INTENSITY_Matrix.csv')
ALPHA_02_CSV = Path('../Fase2/output_csv/elita_recalculated_0_2.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')
ALPHA_08_CSV = Path('../Fase2/output_csv/elita_recalculated_0_8.csv')
OUTPUT_DIR   = Path('output_confronto')
OUTPUT_DIR.mkdir(exist_ok=True)

BASIC_EMOTIONS = ['gioia','tristezza','rabbia','paura','disgusto','fiducia','sorpresa','aspettativa']
EMOTION_COLORS = {
    'gioia':'#FDD835','tristezza':'#1E88E5','rabbia':'#E53935','paura':'#43A047',
    'disgusto':'#8E24AA','fiducia':'#81C784','sorpresa':'#039BE5',
    'aspettativa':'#FB8C00','neutrale':'#9E9E9E',
}
SEVEN_EMOTIONS  = ['gioia','tristezza','rabbia','paura','disgusto','fiducia','sorpresa']
POSITIVE = {'gioia','fiducia','sorpresa','aspettativa'}
NEGATIVE = {'tristezza','rabbia','paura','disgusto'}

print('Configurazione caricata.')

Configurazione caricata.


## Caricamento corpus, token e matrici ELIta

In [2]:
df_corpus = pd.read_csv(CORPUS_CSV)
df_tokens = pd.read_csv(TOKENS_CSV)
df_tokens['lemma'] = df_tokens['lemma'].astype(str).str.lower().str.strip()
df_tokens['pos']   = df_tokens['pos'].astype(str).str.upper().str.strip()
print('Corpus:', len(df_corpus), 'commenti |', 'Token:', len(df_tokens))

Corpus: 700 commenti | Token: 64812


In [3]:
df_matrix = pd.read_csv(ELITA_CSV, index_col=0)

def is_not_emoji(text):
    return emoji.emoji_count(str(text)) == 0

df_elita_orig = df_matrix[df_matrix.index.map(is_not_emoji)][BASIC_EMOTIONS].fillna(0)
df_elita_orig.index = df_elita_orig.index.astype(str).str.lower().str.strip()

def load_recalc(path):
    df = pd.read_csv(path, index_col=0)
    df.index = df.index.astype(str).str.lower().str.strip()
    return df[BASIC_EMOTIONS].fillna(0)

MATRICES = {
    'Originale (α=0)' : df_elita_orig,
    'Ibrido (α=0.2)'  : load_recalc(ALPHA_02_CSV),
    'Ibrido (α=0.5)'  : load_recalc(ALPHA_05_CSV),
    'Ibrido (α=0.8)'  : load_recalc(ALPHA_08_CSV),
}
print('Matrici:', list(MATRICES.keys()))

Matrici: ['Originale (α=0)', 'Ibrido (α=0.2)', 'Ibrido (α=0.5)', 'Ibrido (α=0.8)']


## Funzione base e prima analisi (raw)

Per ogni commento sommiamo i vettori emotivi di tutti i lemmi ADJ+NOUN+VERB trovati in ELIta.
Nessun filtro, nessuna normalizzazione.

In [4]:
def detect_emotions(df_corpus, df_tokens, df_elita, emotions=None):
    if emotions is None:
        emotions = BASIC_EMOTIONS
    pos_filter = {'ADJ','NOUN','VERB'}
    df_f = df_tokens[df_tokens['pos'].isin(pos_filter)].copy()
    eidx = set(df_elita.index)
    tok  = df_f.groupby('comment_id')['lemma'].apply(list).to_dict()
    results = []
    for _, row in df_corpus.iterrows():
        cid   = row['comment_id']
        lemmi = tok.get(cid, [])
        sc = {e: 0.0 for e in emotions}
        found = 0
        for lemma in lemmi:
            if lemma in eidx:
                found += 1
                for e in emotions:
                    sc[e] += df_elita.loc[lemma, e]
        results.append({'comment_id':cid,'n_tokens_matched':found,**sc,
            'dominant_emotion': max(sc,key=sc.get) if found>0 else 'neutrale'})
    return pd.DataFrame(results)

df_raw     = detect_emotions(df_corpus, df_tokens, df_elita_orig)
counts_raw = df_raw['dominant_emotion'].value_counts()
total      = len(df_raw)

df_raw.to_csv(OUTPUT_DIR / 'notizie_emotion_results_raw.csv', index=False)
print(f'Salvato: {OUTPUT_DIR / "1_notizie_emotion_results_raw.csv"}')

print('Distribuzione emozione dominante — raw:')
for e in BASIC_EMOTIONS + ['neutrale']:
    n = counts_raw.get(e,0)
    print('{:<15s} {:>4d} ({:>4.1f}%) {}'.format(e, n, n/total*100, '█'*int(n/total*40)))

Salvato: output_confronto/1_notizie_emotion_results_raw.csv
Distribuzione emozione dominante — raw:
gioia             24 ( 3.4%) █
tristezza          3 ( 0.4%) 
rabbia             4 ( 0.6%) 
paura             11 ( 1.6%) 
disgusto           0 ( 0.0%) 
fiducia            1 ( 0.1%) 
sorpresa          44 ( 6.3%) ██
aspettativa      612 (87.4%) ██████████████████████████████████
neutrale           1 ( 0.1%) 


In [5]:
fig = go.Figure()
for e in BASIC_EMOTIONS + ['neutrale']:
    n = counts_raw.get(e, 0)
    fig.add_trace(go.Bar(name=e, x=[e], y=[round(n/total*100,1)],
        marker_color=EMOTION_COLORS.get(e,'#999'),
        text=['{:.0f}%'.format(n/total*100)], textposition='outside'))
fig.update_layout(title='Distribuzione emozione dominante — analisi raw',
                  barmode='group', height=450, showlegend=False)
fig.show()

**Osservazione**: aspettativa domina massicciamente (~87%).
Come documentato in ItEm (Pollacci 2015), alcune emozioni producono coseni più alti
diventando "catalizzanti". Il primo passo è capire se questo bias è strutturale o semantico.
Seguiamo l'approccio di ItEm: ripetiamo l'analisi escludendo aspettativa.

## Passo 1 — Rimozione di aspettativa (corpus_7emo)

Come `corpus_sei_emo` in ItEm (che escludeva fiducia e attese), escludiamo aspettativa
per vedere la struttura emotiva sottostante. Non è il metodo finale — serve a esplorare.

In [6]:
df_7emo    = detect_emotions(df_corpus, df_tokens, df_elita_orig, emotions=SEVEN_EMOTIONS)
counts_7   = df_7emo['dominant_emotion'].value_counts()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Raw (8 emozioni)','Senza aspettativa (7 emozioni)'],
    horizontal_spacing=0.1)
for col_idx,(df_r,emos) in enumerate([(df_raw,BASIC_EMOTIONS),(df_7emo,SEVEN_EMOTIONS)],start=1):
    counts = df_r['dominant_emotion'].value_counts()
    for e in emos+['neutrale']:
        n = counts.get(e,0)
        fig.add_trace(go.Bar(name=e, x=[e], y=[round(n/total*100,1)],
            marker_color=EMOTION_COLORS.get(e,'#999'),
            showlegend=(col_idx==1), legendgroup=e,
            text=['{:.0f}%'.format(n/total*100)], textposition='outside'),
            row=1, col=col_idx)
fig.update_layout(title='Raw vs corpus_7emo (senza aspettativa)',
                  barmode='group', height=500)
fig.show()

print('Senza aspettativa emergono sorpresa ({:.0f}%) e gioia ({:.0f}%)'.format(
    counts_7.get('sorpresa',0)/total*100, counts_7.get('gioia',0)/total*100))

Senza aspettativa emergono sorpresa (27%) e gioia (48%)


**Osservazione**: senza aspettativa emergono sorpresa e gioia come dominanti.
Questa soluzione però è artificiosa: rimuove informazione invece di correggere il bias.
L'approccio corretto secondo ItEm è la normalizzazione.

## Passo 2 — Normalizzazione corpus_mean (Formula 3.5 di ItEm)

ItEm propone il `corpus_mean` per eliminare il vantaggio sistematico dei coseni alti.
La Formula 3.5 normalizza **per parola**: per ogni termine, il suo score per emozione `e`
viene diviso per la somma degli score su tutte le 8 emozioni per quel termine.

```
score_norm(e, parola) = score(e, parola) / Σ score(e', parola)
S_e(commento)        = Σ score_norm(e, parola)
```

Ogni parola contribuisce con peso proporzionale alla sua unicità emotiva.

In [7]:
def detect_emotions_mean(df_corpus, df_tokens, df_elita):
    """corpus_mean Formula 3.5 ItEm: normalizzazione per parola prima di sommare."""
    pos_filter = {'ADJ','NOUN','VERB'}
    df_f = df_tokens[df_tokens['pos'].isin(pos_filter)].copy()
    eidx = set(df_elita.index)
    tok  = df_f.groupby('comment_id')['lemma'].apply(list).to_dict()
    results = []
    for _, row in df_corpus.iterrows():
        cid   = row['comment_id']
        lemmi = tok.get(cid, [])
        sc = {e: 0.0 for e in BASIC_EMOTIONS}
        found = 0
        for lemma in lemmi:
            if lemma in eidx:
                found += 1
                wv    = {e: df_elita.loc[lemma,e] for e in BASIC_EMOTIONS}
                wtot  = sum(wv.values())
                if wtot > 0:
                    for e in BASIC_EMOTIONS:
                        sc[e] += wv[e] / wtot
        results.append({'comment_id':cid,'n_tokens_matched':found,**sc,
            'dominant_emotion': max(sc,key=sc.get) if found>0 else 'neutrale'})
    return pd.DataFrame(results)

df_mean     = detect_emotions_mean(df_corpus, df_tokens, df_elita_orig)
counts_mean = df_mean['dominant_emotion'].value_counts()

print('Distribuzione — corpus_mean (Formula 3.5 ItEm):')
for e in BASIC_EMOTIONS + ['neutrale']:
    n  = counts_mean.get(e,0)
    n0 = counts_raw.get(e,0)
    print('{:<15s} {:>4d} ({:>4.1f}%)  [{:+d} vs raw]'.format(e,n,n/total*100,n-n0))

Distribuzione — corpus_mean (Formula 3.5 ItEm):
gioia             26 ( 3.7%)  [+2 vs raw]
tristezza          5 ( 0.7%)  [+2 vs raw]
rabbia             1 ( 0.1%)  [-3 vs raw]
paura              5 ( 0.7%)  [-6 vs raw]
disgusto           2 ( 0.3%)  [+2 vs raw]
fiducia            2 ( 0.3%)  [+1 vs raw]
sorpresa          32 ( 4.6%)  [-12 vs raw]
aspettativa      626 (89.4%)  [+14 vs raw]
neutrale           1 ( 0.1%)  [+0 vs raw]


In [8]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Raw','corpus_mean (ItEm Formula 3.5)'],
    horizontal_spacing=0.1)
for col_idx, df_r in enumerate([df_raw, df_mean], start=1):
    counts = df_r['dominant_emotion'].value_counts()
    for e in BASIC_EMOTIONS + ['neutrale']:
        n = counts.get(e,0)
        fig.add_trace(go.Bar(name=e, x=[e], y=[round(n/total*100,1)],
            marker_color=EMOTION_COLORS.get(e,'#999'),
            showlegend=(col_idx==1), legendgroup=e,
            text=['{:.0f}%'.format(n/total*100)], textposition='outside'),
            row=1, col=col_idx)
fig.update_layout(title='Raw vs corpus_mean — aspettativa NON scende',
                  barmode='group', height=500)
fig.show()

**Risultato inatteso**: corpus_mean **fa salire** aspettativa (da 87% a 89%).

**Perché succede**: in ItEm gli score sono coseni distribuzionali — valori continui, quasi mai zero su tutte le 8 emozioni. Con quelle distribuzioni la normalizzazione per parola riduce correttamente il vantaggio delle emozioni con coseni alti.

In ELIta le annotazioni sono discrete (0 / 0.25 / 0.75 / 1). Molte parole hanno 0
su 7 emozioni e solo un piccolo valore su aspettativa:

```
[0, 0, 0, 0, 0, 0, 0, 0.25]  →  normalizzazione per parola  →  [0, 0, 0, 0, 0, 0, 0, 1.0]
```

Quella parola viene **amplificata a 1.0** su aspettativa. La Formula 3.5 di ItEm
non è direttamente trasferibile su ELIta perché lo schema di annotazione è diverso.

Il bias persiste e anzi peggiora: dobbiamo capire da dove viene davvero.

In [9]:
# Verifichiamo quante parole di ELIta hanno aspettativa come unica emozione
nonzero = (df_elita_orig > 0).sum(axis=1)
only_asp   = df_elita_orig[(nonzero == 1) & (df_elita_orig['aspettativa'] > 0)]
sparse_asp = df_elita_orig[(nonzero <= 2) & (df_elita_orig['aspettativa'] > 0)]

print('Parole con aspettativa come unica emozione non-zero: {:d}'.format(len(only_asp)))
print('Parole con aspettativa non-zero e ≤ 2 emozioni totali: {:d}'.format(len(sparse_asp)))
print()
print('Con norma per-parola, ogni occorrenza di queste parole diventa aspettativa=1.0')
print('=> la norma PER PAROLA AMPLIFICA le parole sparse di ELIta.')
print()
print('La Formula 3.5 di ItEm richiede score continui su tutte le emozioni.')
print('Con annotazioni discrete e sparse (come ELIta) il risultato è opposto.')

Parole con aspettativa come unica emozione non-zero: 7
Parole con aspettativa non-zero e ≤ 2 emozioni totali: 34

Con norma per-parola, ogni occorrenza di queste parole diventa aspettativa=1.0
=> la norma PER PAROLA AMPLIFICA le parole sparse di ELIta.

La Formula 3.5 di ItEm richiede score continui su tutte le emozioni.
Con annotazioni discrete e sparse (come ELIta) il risultato è opposto.


## Diagnosi: cosa guida aspettativa nel raw?

Dato che la normalizzazione non aiuta, analizziamo le parole che contribuiscono
di più ad aspettativa per identificare il problema reale.

In [10]:
POS_FILTER = {'ADJ','NOUN','VERB'}
df_filt = df_tokens[df_tokens['pos'].isin(POS_FILTER)].copy()
elita_idx = set(df_elita_orig.index)

matched = sorted(set(df_filt['lemma']).intersection(elita_idx))
freq    = df_filt[df_filt['lemma'].isin(matched)]['lemma'].value_counts()

freq_df = freq.reset_index()
freq_df.columns = ['lemma','frequenza']
er = df_elita_orig.loc[matched, BASIC_EMOTIONS].reset_index()
er.columns = ['lemma'] + BASIC_EMOTIONS
freq_df = freq_df.merge(er, on='lemma', how='left')
freq_df['word_sum'] = freq_df[BASIC_EMOTIONS].sum(axis=1)
freq_df['contrib_asp'] = freq_df['frequenza'] * freq_df['aspettativa']

print('Top 25 parole per contributo ad ASPETTATIVA (frequenza × score):')
display(freq_df.nlargest(25,'contrib_asp')[['lemma','frequenza','aspettativa','word_sum','contrib_asp']].round(3).reset_index(drop=True))

Top 25 parole per contributo ad ASPETTATIVA (frequenza × score):


,lemma,frequenza,aspettativa,word_sum,contrib_asp
0,notizia,810,0.71,3.12,575.10
1,fare,584,0.58,1.32,338.72
2,avere,387,0.58,2.86,224.46
3,vedere,199,0.54,2.63,107.46
4,dire,226,0.38,1.93,85.88
5,dare,112,0.71,2.76,79.52
6,pensare,98,0.75,3.75,73.50
7,trovare,75,0.92,4.04,69.00
8,anno,127,0.54,1.79,68.58
9,donna,88,0.75,2.37,66.00


**Osservazione**: tra le parole che più contribuiscono ad aspettativa troviamo
verbi ausiliari e nomi generici (*avere*, *fare*, *cosa*, *anno*, *modo*...).

Queste parole sono **frequentissime in qualsiasi testo italiano** e hanno score
non nullo su aspettativa in ELIta. Ma nel testo non esprimono nessuna emozione
specifica: dire *avere* o *fare* non comunica aspettativa.

Il problema non è strutturale (coseni alti) — è **semantico**: parole semanticamente
vuote che compaiono centinaia di volte e accumulano un piccolo contributo ad aspettativa.

## Passo 3 — EMOTIONAL_STOPWORDS

La soluzione è escludere i lemmi frequenti ma privi di contenuto emotivo specifico
nel contesto dell'analisi su testo libero.

In [11]:
EMOTIONAL_STOPWORDS = {
     # Verbi ausiliari / modali / supporto
    'avere','essere','fare','stare','dare','andare','venire',
    'potere','volere','dovere','sapere','vedere','sentire',
    'trovare','pensare','dire','parlare','guardare','tenere',
    'portare','prendere','mettere','lasciare','passare','uscire',
    'entrare','tornare','rimanere','iniziare','finire','continuare',
    'cominciare','provare','riuscire','sembrare','diventare',
    # Nomi generici / funzionali
    'cosa','modo','parte','punto','volta','anno','tempo','caso',
    'fatto','posto','tipo','gente','persona','vita','mondo',
    'uomo','donna','bambino','figlio','figlia','padre','madre',
    # Aggettivi generici
    'altro','solo','grande','piccolo','nuovo','vecchio','primo',
    'ultimo','stesso','proprio','bello','buono','lungo','alto',
    # Quantificatori
    'più','bene','male','molto','poco','tanto','tutto','niente',
}

in_elita   = [w for w in EMOTIONAL_STOPWORDS if w in elita_idx]
freq_stop  = df_filt[df_filt['lemma'].isin(in_elita)]['lemma'].value_counts()
freq_total = len(df_filt)

print('Stopwords definite: {:d} | Presenti in ELIta: {:d}'.format(
    len(EMOTIONAL_STOPWORDS), len(in_elita)))
print('Token rimossi: {:d}/{:d} ({:.1f}%)'.format(
    freq_stop.sum(), freq_total, freq_stop.sum()/freq_total*100))
print()

Stopwords definite: 80 | Presenti in ELIta: 75
Token rimossi: 5017/26821 (18.7%)



In [12]:
# Quante delle top-25 driver di aspettativa sono stopwords?
top25 = set(freq_df.nlargest(25,'contrib_asp')['lemma'])
overlap = top25 & EMOTIONAL_STOPWORDS
print('Driver di aspettativa (top-25) che sono stopwords: {:d}/25'.format(len(overlap)))
print(sorted(overlap))

Driver di aspettativa (top-25) che sono stopwords: 18/25
['anno', 'avere', 'dare', 'dire', 'donna', 'fare', 'fatto', 'mondo', 'parlare', 'pensare', 'sentire', 'tempo', 'tipo', 'trovare', 'uomo', 'vedere', 'vita', 'volere']


In [13]:
# Parole che definiscono il topic "notizie", non l'emozione.
# "notizia" da sola vale 575 punti ad aspettativa (810 occorrenze × 0.71):
# è la keyword di ricerca, compare per definizione ovunque nel corpus.
TOPIC_STOPWORDS = {
    'notizia', 'notizie', 'giornale', 'giornali', 'giornalista', 'giornalismo',
    'informazione', 'informazioni', 'articolo', 'articoli', 'media', 'fonte',
    'fonti', 'testata', 'redazione', 'titolo', 'telegiornale',
}

ALL_STOPWORDS = EMOTIONAL_STOPWORDS | TOPIC_STOPWORDS

in_elita_t = [w for w in TOPIC_STOPWORDS if w in elita_idx]
print('Topic stopwords aggiunte: {:d} | Presenti in ELIta: {:d}'.format(
    len(TOPIC_STOPWORDS), len(in_elita_t)))
print()
# Aggiorniamo anche il messaggio sulle stopwords generali
in_elita   = [w for w in ALL_STOPWORDS if w in elita_idx]
freq_stop  = df_filt[df_filt['lemma'].isin(in_elita)]['lemma'].value_counts()
print('Token rimossi con ALL_STOPWORDS: {:d}/{:d} ({:.1f}%)'.format(
    freq_stop.sum(), freq_total, freq_stop.sum()/freq_total*100))

# Quante delle top-25 driver di aspettativa sono ora coperte?
top25 = set(freq_df.nlargest(25,'contrib_asp')['lemma'])
overlap_all = top25 & ALL_STOPWORDS
print('Driver di aspettativa (top-25) coperti da ALL_STOPWORDS: {:d}/25'.format(len(overlap_all)))
print(sorted(overlap_all))

Topic stopwords aggiunte: 17 | Presenti in ELIta: 10

Token rimossi con ALL_STOPWORDS: 6252/26821 (23.3%)
Driver di aspettativa (top-25) coperti da ALL_STOPWORDS: 21/25
['anno', 'avere', 'dare', 'dire', 'donna', 'fare', 'fatto', 'giornale', 'informazione', 'mondo', 'notizia', 'parlare', 'pensare', 'sentire', 'tempo', 'tipo', 'trovare', 'uomo', 'vedere', 'vita', 'volere']


### Scelta delle soglie: distribuzione di dist_fase2

Prima di testare le soglie, analizziamo come sono distribuiti i valori di distintività
su tutto il lessico ELIta. Questo permette di scegliere soglie radicate nella struttura
del dato, non arbitrarie.

In [33]:
def calculate_distinctiveness_fase2(df_elita):
    """Formula di distintività da Fase2/Valutazione e Ricalcolo.ipynb (ItEm).
    Per ogni parola calcola quanto è 'pura' rispetto all'emozione dominante:
        d = ((max1 - max2) / max1) * (max1 - mean_emozioni)
    Restituisce un Series con il valore massimo di d per parola (emozione dominante).
    """
    def _row_dist(row):
        sv   = sorted(row.values, reverse=True)
        max1, max2 = sv[0], sv[1]
        mn   = np.mean(row.values)
        if max1 == 0:
            return 0.0
        return ((max1 - max2) / (max1 + 1e-9)) * (max1 - mn)

    return df_elita[BASIC_EMOTIONS].apply(_row_dist, axis=1)


def detect_emotions_sw(df_corpus, df_tokens, df_elita, stopwords,
                        dist_threshold=0.0):
    """Emotion detection con stopwords e soglia di distintività (formula Fase2).
    dist_threshold: soglia sulla distintività calcolata con la formula ItEm
    (stessa usata in Fase2 per selezionare le parole seme dei centroidi)."""
    pos_filter = {'ADJ', 'NOUN', 'VERB'}
    df_f = df_tokens[df_tokens['pos'].isin(pos_filter)].copy()
    df_f = df_f[~df_f['lemma'].isin(stopwords)]
    eidx = set(df_elita.index)

    if dist_threshold > 0:
        dist_scores = calculate_distinctiveness_fase2(df_elita)
        eidx = eidx & set(dist_scores[dist_scores >= dist_threshold].index)

    tok = df_f.groupby('comment_id')['lemma'].apply(list).to_dict()
    results = []
    for _, row in df_corpus.iterrows():
        cid   = row['comment_id']
        lemmi = tok.get(cid, [])
        sc    = {e: 0.0 for e in BASIC_EMOTIONS}
        found = 0
        for lemma in lemmi:
            if lemma in eidx:
                found += 1
                for e in BASIC_EMOTIONS:
                    sc[e] += df_elita.loc[lemma, e]
        results.append({'comment_id': cid, 'n_tokens_matched': found, **sc,
            'dominant_emotion': max(sc, key=sc.get) if found > 0 else 'neutrale'})
    return pd.DataFrame(results)

# Prima applicazione con ALL_STOPWORDS (senza soglia)
df_sw     = detect_emotions_sw(df_corpus, df_tokens, df_elita_orig, ALL_STOPWORDS)
counts_sw = df_sw['dominant_emotion'].value_counts()

print('Distribuzione — ALL_STOPWORDS (senza soglia):')
for e in BASIC_EMOTIONS + ['neutrale']:
    n  = counts_sw.get(e, 0)
    n0 = counts_raw.get(e, 0)
    print('{:<15s} {:>4d} ({:>4.1f}%)  [{:+d} vs raw]'.format(e, n, n/total*100, n-n0))


Distribuzione — ALL_STOPWORDS (senza soglia):
gioia             76 (10.9%)  [+52 vs raw]
tristezza         26 ( 3.7%)  [+23 vs raw]
rabbia            27 ( 3.9%)  [+23 vs raw]
paura             35 ( 5.0%)  [+24 vs raw]
disgusto           4 ( 0.6%)  [+4 vs raw]
fiducia           23 ( 3.3%)  [+22 vs raw]
sorpresa          14 ( 2.0%)  [-30 vs raw]
aspettativa      484 (69.1%)  [-128 vs raw]
neutrale          11 ( 1.6%)  [+10 vs raw]


### Soglia di distintività

Molte parole hanno score simili su più emozioni (gap piccolo tra 1° e 2° emozione).
Includere solo parole con una chiara emozione dominante riduce il rumore.

Il trade-off: soglia più alta → meno aspettativa ma meno copertura del corpus.

In [36]:
# Distribuzione dei valori di dist_fase2 su tutto il lessico
dist_all = calculate_distinctiveness_fase2(df_elita_orig)

print('Distribuzione dist_fase2 — lessico ELIta:')
print(dist_all.describe().round(4).to_string())

Distribuzione dist_fase2 — lessico ELIta:
count    6719.0000
mean        0.0838
std         0.0932
min         0.0000
25%         0.0206
50%         0.0538
75%         0.1173
max         0.7255


In [37]:
print('Percentili:')
print('{:<6s} {:>10s} {:>15s}'.format('p', 'soglia', 'parole incluse'))
print('-' * 35)
for p in [25, 40, 50, 60, 70, 75, 80, 90]:
    v = dist_all.quantile(p/100)
    n = (dist_all >= v).sum()
    print('p{:<4d} {:>10.4f} {:>12d}'.format(p, v, n))

Percentili:
p          soglia  parole incluse
-----------------------------------
p25       0.0206         5039
p40       0.0394         4031
p50       0.0538         3360
p60       0.0741         2688
p70       0.1003         2016
p75       0.1173         1680
p80       0.1400         1344
p90       0.2016          673


In [41]:
# Confronto soglie di distintività — formula Fase2
# La formula è: d = ((max1 - max2) / max1) * (max1 - mean)
# stessa usata per selezionare i semi dei centroidi in Valutazione e Ricalcolo.ipynb

configs_dist = [
    ('ALL_STOPWORDS (dist=0)',     ALL_STOPWORDS, 0.0),
    ('+ dist_fase2 ≥ 0.04',        ALL_STOPWORDS, 0.04),
    ('+ dist_fase2 ≥ 0.06',        ALL_STOPWORDS, 0.06),
    ('+ dist_fase2 ≥ 0.08',        ALL_STOPWORDS, 0.08),
    ('+ dist_fase2 ≥ 0.10',        ALL_STOPWORDS, 0.10),
]

print('{:<35s} | {:>6s} | {:>6s} | {:>6s} | {:>7s}'.format(
    'Configurazione', 'asp', 'gioia', 'rabbia', 'match%'))
print('-' * 75)
for label, sw, dist in configs_dist:
    df_r = detect_emotions_sw(df_corpus, df_tokens, df_elita_orig, sw, dist)
    c    = df_r['dominant_emotion'].value_counts()
    m    = (df_r['n_tokens_matched'] > 0).sum()
    print('{:<35s} | {:>5.1f}% | {:>5.1f}% | {:>5.1f}% | {:>6.0f}%'.format(
        label,
        c.get('aspettativa', 0)/total*100,
        c.get('gioia', 0)/total*100,
        c.get('rabbia', 0)/total*100,
        m/total*100))
print()
print('Scegliamo dist_fase2 ≥ 0.06 (equivalente a gap ≥ 0.10, ~3400 parole, copertura 90%).')


Configurazione                      |    asp |  gioia | rabbia |  match%
---------------------------------------------------------------------------
ALL_STOPWORDS (dist=0)              |  69.1% |  10.9% |   3.9% |     98%
+ dist_fase2 ≥ 0.04                 |  61.6% |  13.1% |   4.7% |     95%
+ dist_fase2 ≥ 0.06                 |  54.3% |  15.9% |   3.9% |     90%
+ dist_fase2 ≥ 0.08                 |  49.7% |  17.1% |   3.6% |     87%
+ dist_fase2 ≥ 0.10                 |  42.6% |  18.9% |   3.9% |     83%

Scegliamo dist_fase2 ≥ 0.06 (equivalente a gap ≥ 0.10, ~3400 parole, copertura 90%).


In [43]:
# Confronto progressione: raw → mean → sw
fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Raw','corpus_mean','Solo stopwords'],
    horizontal_spacing=0.08)
for col_idx, df_r in enumerate([df_raw, df_mean, df_sw], start=1):
    counts = df_r['dominant_emotion'].value_counts()
    emos = BASIC_EMOTIONS if col_idx <= 2 else BASIC_EMOTIONS
    for e in emos + ['neutrale']:
        n = counts.get(e,0)
        fig.add_trace(go.Bar(name=e, x=[e[:4]], y=[round(n/total*100,1)],
            marker_color=EMOTION_COLORS.get(e,'#999'),
            showlegend=(col_idx==1), legendgroup=e,
            text=['{:.0f}%'.format(n/total*100)], textposition='outside'),
            row=1, col=col_idx)
fig.update_layout(title='Progressione: Raw → corpus_mean → Stopwords',
                  barmode='group', height=500)
fig.show()

In [42]:
# Bilancio pos/neg
print('{:<25s} | {:>10s} | {:>10s}'.format('Configurazione','Positive','Negative'))
print('-'*52)
for label, df_r in [('Raw',df_raw),('corpus_mean',df_mean),('Stopwords',df_sw)]:
    c = df_r['dominant_emotion'].value_counts()
    pos = sum(c.get(e,0) for e in POSITIVE)
    neg = sum(c.get(e,0) for e in NEGATIVE)
    print('{:<25s} | {:>9.1f}% | {:>9.1f}%'.format(label,pos/total*100,neg/total*100))

Configurazione            |   Positive |   Negative
----------------------------------------------------
Raw                       |      97.3% |       2.6%
corpus_mean               |      98.0% |       1.9%
Stopwords                 |      85.3% |      13.1%


## Metodo finale: stopwords su tutte le versioni ELIta
Qui si applica il metodo finale — ALL_STOPWORDS + dist_fase2 ≥ 0.06 — a tutte e 4 le versioni di ELIta, non solo all'originale.

In [44]:
DIST_THRESHOLD = 0.06

results_final = {}
for vname, df_e in MATRICES.items():
    df_r = detect_emotions_sw(df_corpus, df_tokens, df_e,
                               ALL_STOPWORDS, dist_threshold=DIST_THRESHOLD)
    results_final[vname] = df_r
    matched = (df_r['n_tokens_matched'] > 0).sum()
    print('{:<20s} | match: {:d}/{:d} ({:.0f}%)'.format(
          vname, matched, len(df_r), matched/len(df_r)*100))

Originale (α=0)      | match: 633/700 (90%)
Ibrido (α=0.2)       | match: 625/700 (89%)
Ibrido (α=0.5)       | match: 612/700 (87%)
Ibrido (α=0.8)       | match: 573/700 (82%)


In [62]:
fig = make_subplots(rows=2, cols=2, subplot_titles=list(MATRICES.keys()),
    vertical_spacing=0.18, horizontal_spacing=0.08)
positions = [(1,1),(1,2),(2,1),(2,2)]
for idx,(vname,df_r) in enumerate(results_final.items()):
    counts = df_r['dominant_emotion'].value_counts()
    r,c = positions[idx]
    for e in BASIC_EMOTIONS + ['neutrale']:
        n = counts.get(e,0)
        fig.add_trace(go.Bar(name=e, x=[e], y=[round(n/total*100,1)],
            marker_color=EMOTION_COLORS.get(e,'#999'),
            showlegend=(idx==0), legendgroup=e,
            text=['{:.0f}%'.format(n/total*100)], textposition='outside'),
            row=r, col=c)
fig.update_layout(title='Distribuzione emozione dominante — metodo finale (stopwords)',
                  barmode='group', height=700)
fig.show()

## Confronto quantitativo: originale vs ricalcolato

L'emozione dominante è winner-takes-all. Misuriamo l'impatto del ricalcolo
con metriche continue: score medi, gap 1°-2°, Silhouette score.

In [53]:
print('Score emotivi medi per versione:')
print('{:<20s}'.format('Emozione'), end='')
for vname in MATRICES: print(' {:>20s}'.format(vname[:20]), end='')
print()
print('-'*100)
means_table = {}
for e in BASIC_EMOTIONS:
    print('{:<20s}'.format(e), end='')
    means_table[e] = {}
    for vname in MATRICES:
        m = results_final[vname][e].mean()
        means_table[e][vname] = m
        print(' {:>18.4f}'.format(m), end='')
    print()

Score emotivi medi per versione:
Emozione                  Originale (α=0)       Ibrido (α=0.2)       Ibrido (α=0.5)       Ibrido (α=0.8)
----------------------------------------------------------------------------------------------------
gioia                            2.4815             2.5366             2.5228             2.3172
tristezza                        1.5147             1.6456             1.6614             1.5709
rabbia                           1.3657             1.5397             1.5850             1.5108
paura                            1.6355             1.8185             1.8542             1.7575
disgusto                         0.8281             1.0412             1.1735             1.1685
fiducia                          2.3668             2.5677             2.7399             2.6163
sorpresa                         1.8485             2.1359             2.4184             2.4166
aspettativa                      3.2688             3.2988             3.2375     

Ogni numero rappresenta la somma media degli score emotivi per commento — non una percentuale, ma un valore assoluto che dipende da quante parole vengono trovate e quanto sono intense emotivamente.

Notiamo che la versione con α=0.5 ha score medi più alti su tutte le emozioni, riducendo il gap tra aspettativa e le altre emozioni.

In [58]:
print('Variazione vs originale:')
print('{:<20s}'.format('Emozione'), end='')
for vname in list(MATRICES.keys())[1:]: print(' {:>20s}'.format(vname[:20]), end='')
print()
print('-'*100)
for e in BASIC_EMOTIONS:
    print('{:<20s}'.format(e), end='')
    base = means_table[e]['Originale (α=0)']
    for vname in list(MATRICES.keys())[1:]:
        print(' {:>+18f}'.format(means_table[e][vname]-base), end='')
    print()

Variazione vs originale:
Emozione                   Ibrido (α=0.2)       Ibrido (α=0.5)       Ibrido (α=0.8)
----------------------------------------------------------------------------------------------------
gioia                         +0.055076          +0.041297          -0.164331
tristezza                     +0.130914          +0.146653          +0.056203
rabbia                        +0.174006          +0.219299          +0.145097
paura                         +0.182967          +0.218681          +0.121959
disgusto                      +0.213126          +0.345486          +0.340487
fiducia                       +0.200860          +0.373090          +0.249479
sorpresa                      +0.287366          +0.569930          +0.568040
aspettativa                   +0.029967          -0.031315          -0.342800


Qui si vedono ancora più chiaramente gli effetti di normalizzazione e stopwords. Tutti valori sono riferiti alla versione originale (α=0).

In [21]:
print('Gap medio e Silhouette:')
print('{:<25s} {:>12s} {:>12s}'.format('Versione','Gap medio','Silhouette'))
print('-'*52)
gap_stats = {}
silhouette_scores = {}
for vname in MATRICES:
    df_r = results_final[vname]
    ss   = df_r[BASIC_EMOTIONS].apply(lambda r: sorted(r.values,reverse=True), axis=1)
    gap  = ss.apply(lambda s: s[0]-s[1])
    gap_nz = gap[gap>0]
    gap_stats[vname] = gap_nz.mean()
    df_v = df_r[(df_r['n_tokens_matched']>0)&(df_r['dominant_emotion']!='neutrale')].copy()
    vc   = df_v['dominant_emotion'].value_counts()
    df_v = df_v[df_v['dominant_emotion'].isin(vc[vc>=2].index)]
    if len(df_v)>=10 and df_v['dominant_emotion'].nunique()>=2:
        sil = silhouette_score(df_v[BASIC_EMOTIONS].values,
                               df_v['dominant_emotion'].values, metric='cosine')
        silhouette_scores[vname] = sil
        print('{:<25s} {:>12.4f} {:>12.4f}'.format(vname, gap_nz.mean(), sil))
    else:
        print('{:<25s} {:>12.4f} {:>12s}'.format(vname, gap_nz.mean(), 'N/A'))
if silhouette_scores:
    best = max(silhouette_scores, key=silhouette_scores.get)
    print('\nVersione con Silhouette migliore:', best)

Gap medio e Silhouette:
Versione                     Gap medio   Silhouette
----------------------------------------------------
Originale (α=0)                 0.7689       0.1909
Ibrido (α=0.2)                  0.6708       0.1848
Ibrido (α=0.5)                  0.5473       0.1629
Ibrido (α=0.8)                  0.4259       0.1274

Versione con Silhouette migliore: Originale (α=0)


In [22]:
means_data = [
    {'Versione':vname,'Emozione':e.capitalize(),'Score medio':results_final[vname][e].mean()}
    for vname in MATRICES for e in BASIC_EMOTIONS
]
fig = px.bar(pd.DataFrame(means_data), x='Emozione', y='Score medio',
    color='Versione', barmode='group',
    title='Score emotivo medio per versione ELIta — corpus notizie (metodo finale)',
    color_discrete_sequence=['#455A64','#1E88E5','#FB8C00','#E53935'], text_auto='.3f')
fig.update_traces(textposition='outside', textfont_size=9)
fig.update_layout(height=500)
fig.show()

Possiamo vedere che la versione con α=0.5 è quella a noi più conveniente: riduce il gap tra aspettativa e le altre emozioni.

## 10. Tabella riassuntiva

In [59]:
# Tabella A: N. commenti e token con score > 0 per emozione
df_filt_sw = df_filt[~df_filt['lemma'].isin(ALL_STOPWORDS)]
eidx_dist = set(df_elita_orig.index)
if DIST_THRESHOLD > 0:
    sorted_sc = df_elita_orig[BASIC_EMOTIONS].apply(lambda r: sorted(r.values,reverse=True), axis=1)
    dist_col  = sorted_sc.apply(lambda s: s[0]-s[1] if len(s)>1 else s[0])
    eidx_dist = eidx_dist & set(df_elita_orig[dist_col >= DIST_THRESHOLD].index)

# Tabella A: usa df_sw (senza norma, per conteggi significativi)
df_sw_forA = detect_emotions_sw(df_corpus, df_tokens, df_elita_orig, ALL_STOPWORDS, 0.0)
table_A = []
for e in BASIC_EMOTIONS:
    comm = df_sw_forA[df_sw_forA[e]>0]['comment_id']
    ntok = df_filt_sw[df_filt_sw['comment_id'].isin(comm)&df_filt_sw['lemma'].isin(eidx_dist)]['lemma'].count()
    table_A.append({'Emozione':e.capitalize(),
                    'N. Commenti (score>0)':int((df_sw_forA[e]>0).sum()),
                    'N. Token':int(ntok)})
print('Tabella A — N. commenti e token per emozione (metodo finale):')
display(pd.DataFrame(table_A))

Tabella A — N. commenti e token per emozione (metodo finale):


,Emozione,N. Commenti (score>0),N. Token
0,Gioia,684,8265
1,Tristezza,681,8259
2,Rabbia,683,8264
3,Paura,679,8254
4,Disgusto,674,8252
5,Fiducia,686,8264
6,Sorpresa,687,8266
7,Aspettativa,688,8266


In [61]:
dom = results_final['Originale (α=0)']['dominant_emotion'].value_counts()
tot = len(results_final['Originale (α=0)'])
table_B = [{'Emozione':e.capitalize(),
            'N. Commenti dom':int(dom.get(e,0)),
            '% totale':'{:.1f}%'.format(dom.get(e,0)/tot*100)}
           for e in BASIC_EMOTIONS+['neutrale']]
print('\nTabella B — Emozione dominante (ALL_STOPWORDS + dist_fase2 ≥ 0.06):')
display(pd.DataFrame(table_B))


Tabella B — Emozione dominante (ALL_STOPWORDS + dist_fase2 ≥ 0.06):


,Emozione,N. Commenti dom,% totale
0,Gioia,111,15.9%
1,Tristezza,27,3.9%
2,Rabbia,27,3.9%
3,Paura,34,4.9%
4,Disgusto,9,1.3%
5,Fiducia,32,4.6%
6,Sorpresa,13,1.9%
7,Aspettativa,380,54.3%
8,Neutrale,67,9.6%


## Salvataggio output

In [24]:
for vname, df_r in results_final.items():
    safe = vname.replace(' ','_').replace('(','').replace(')','').replace('=','')
    df_r.to_csv(OUTPUT_DIR / 'notizie_emotion_results_{}.csv'.format(safe), index=False)
    print('Salvato:', safe)

metrics_rows = []
for vname in MATRICES:
    df_r   = results_final[vname]
    ss     = df_r[BASIC_EMOTIONS].apply(lambda r: sorted(r.values,reverse=True), axis=1)
    gap    = ss.apply(lambda s: s[0]-s[1])
    gap_nz = gap[gap>0]
    row = {'Versione':vname,
           'Gap medio':round(gap_nz.mean(),4),
           'Gap mediana':round(gap_nz.median(),4),
           'Silhouette':round(silhouette_scores.get(vname,float('nan')),4)}
    for e in BASIC_EMOTIONS:
        row['mean_'+e] = round(df_r[e].mean(),4)
    metrics_rows.append(row)
df_metrics = pd.DataFrame(metrics_rows)
df_metrics.to_csv(OUTPUT_DIR / 'notizie_metriche_confronto.csv', index=False)
display(df_metrics)
print('\nSalvata: notizie_metriche_confronto.csv')

Salvato: Originale_α0
Salvato: Ibrido_α0.2
Salvato: Ibrido_α0.5
Salvato: Ibrido_α0.8


,Versione,Gap medio,Gap mediana,Silhouette,mean_gioia,mean_tristezza,mean_rabbia,mean_paura,mean_disgusto,mean_fiducia,mean_sorpresa,mean_aspettativa
0,Originale (α=0),0.7689,0.3900,0.1909,2.4815,1.5147,1.3657,1.6355,0.8281,2.3668,1.8485,3.2688
1,Ibrido (α=0.2),0.6708,0.3346,0.1848,2.5366,1.6456,1.5397,1.8185,1.0412,2.5677,2.1359,3.2988
2,Ibrido (α=0.5),0.5473,0.2988,0.1629,2.5228,1.6614,1.5850,1.8542,1.1735,2.7399,2.4184,3.2375
3,Ibrido (α=0.8),0.4259,0.2540,0.1274,2.3172,1.5709,1.5108,1.7575,1.1685,2.6163,2.4166,2.9260



Salvata: notizie_metriche_confronto.csv


## Conclusioni

### Percorso seguito

- **Raw**: la feature aspettativa domina (~87%). Si cerca la causa.
- **Rimozione aspettativa** (corpus_7emo): struttura sottostante — sorpresa 27%, gioia 48%. Utile per l'esplorazione, non come metodo finale.
- **corpus_mean** (Formula 3.5 ItEm): aspettativa *sale* a 89%. La norma per-parola non funziona con ELIta perché amplifica le parole sparse con una sola annotazione non-zero.
- **EMOTIONAL_STOPWORDS**: aspettativa scende a 79%. Le parole generiche (*avere*, *fare*, *cosa*...) erano driver importanti.
- **TOPIC_STOPWORDS**: aspettativa scende a 69%. `notizia` da sola valeva 575 punti (810 occorrenze × 0.71) — la keyword di ricerca era il principale driver residuo. Le parole di dominio (*giornale*, *informazione*, *media*...) definiscono il topic, non l'emozione.
- **Soglia di distintività ≥ 0.6**: aspettativa scende a ~54%. Il 71% dei top driver residui aveva gap < 0.15 — parole emotivamente piatte che accumulano aspettativa per frequenza.

**Metodo finale** (ALL_STOPWORDS + dist ≥ 0.6 + alpha = 0.5): aspettativa ~50%, copertura 90%.

### Bias tematico residuo
Il ~50% di aspettativa riflette genuinamente il dominio *notizie*: le parole portanti (*cercare*, *esistere*, *domanda*, *accordo*, *possibile*...) hanno aspettativa come emozione dominante in ELIta. L'attesa di sviluppi e informazioni è semanticamente caratteristica del linguaggio delle notizie. Non è un artefatto eliminabile.

### Differenza rispetto a ItEm
In ItEm il corpus_mean funziona perché gli score sono coseni distribuzonali continui. Con ELIta (annotazioni discrete) l'intervento efficace è lessicale: rimozione di parole di topic e filtro per distintività emotiva.